# Crawling Sound / Audio dari TikTok
## Proyek Sains Data (PSD) - Web Audio Classification

Notebook ini bertugas untuk mengunduh (*crawling*) file audio dari daftar tautan video TikTok yang berada pada file **`link tiktok.txt`** dan menyimpannya secara terstruktur ke dalam folder **`hasil crawling sound`**.

### Alur Kerja:
1. **Membaca Tautan**: Membaca dan membersihkan tautan video TikTok dari file `link tiktok.txt`.
2. **Ekstraksi Audio**: Mengambil tautan audio (*direct sound stream*) TikTok menggunakan TikWM API tanpa watermark.
3. **Penyimpanan**: Menyimpan file audio `.mp3` ke folder `hasil crawling sound`.
4. **Logging**: Mencatat status unduhan, judul sound, durasi, dan ID ke `log_crawling_sound.csv`.
5. **Pemutaran Sampel**: Memverifikasi file audio yang berhasil diunduh langsung di dalam notebook.

---  
### 1. Import Library & Konfigurasi Path

In [6]:
import os
import re
import time
import requests
import pandas as pd
from IPython.display import Audio, display

# Konfigurasi Path
INPUT_FILE = "link tiktok.txt"
OUTPUT_DIR = "hasil crawling sound"
LOG_FILE = "log_crawling_sound.csv"

# Pastikan folder output sudah tersedia
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[OK] Folder output siap: {os.path.abspath(OUTPUT_DIR)}")

[OK] Folder output siap: c:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Web Audio\hasil crawling sound


---  
### 2. Membaca & Membersihkan Tautan dari `link tiktok.txt`

In [7]:
urls = []

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            # Hilangkan spasi ekstra/typo (misal: 'co m' menjadi 'com')
            cleaned = line.strip().replace(" ", "")
            if cleaned.startswith("http"):
                urls.append(cleaned)
    print(f"[INFO] Berhasil memuat {len(urls)} tautan TikTok dari '{INPUT_FILE}'.\n")
    for idx, u in enumerate(urls[:5], 1):
        print(f"{idx}. {u}")
    if len(urls) > 5:
        print(f"... dan {len(urls) - 5} tautan lainnya.")
else:
    print(f"[ERROR] File '{INPUT_FILE}' tidak ditemukan di direktori kerja!")

[INFO] Berhasil memuat 32 tautan TikTok dari 'link tiktok.txt'.

1. https://vt.tiktok.com/ZSgcVHh6q/
2. https://vt.tiktok.com/ZSgcVPX2a/
3. https://vt.tiktok.com/ZSgcVMHwj/
4. https://vt.tiktok.com/ZSgcqXyV5/
5. https://vt.tiktok.com/ZSgcqcpej/
... dan 27 tautan lainnya.


---  
### 3. Fungsi Downloader Audio TikTok (TikWM API)

In [8]:
def sanitize_filename(name):
    """Membersihkan karakter terlarang pada nama file Windows."""
    if not name:
        return "untitled"
    cleaned = re.sub(r'[\\/*?:"<>|]', "", str(name)).strip()
    return cleaned if cleaned else "sound"

def download_tiktok_sound(url, output_dir=OUTPUT_DIR, index=1):
    """
    Mengambil URL stream audio MP3 dari tautan TikTok dan mengunduhnya ke folder output.
    """
    api_endpoint = "https://tikwm.com/api/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    result = {
        "no": index,
        "url": url,
        "status": "Gagal",
        "filename": None,
        "file_path": None,
        "video_id": None,
        "sound_title": None,
        "author": None,
        "duration_sec": None,
        "message": ""
    }
    
    try:
        response = requests.get(api_endpoint, params={"url": url}, headers=headers, timeout=20)
        if response.status_code != 200:
            result["message"] = f"HTTP status error: {response.status_code}"
            return result
            
        data = response.json()
        if data.get("code") != 0:
            result["message"] = data.get("msg", "Gagal mengekstrak data dari TikTok")
            return result
            
        item = data.get("data", {})
        video_id = str(item.get("id", f"tiktok_{index}"))
        music_info = item.get("music_info", {})
        
        # Ambil direct link audio MP3
        sound_url = item.get("music") or music_info.get("play")
        if not sound_url:
            result["message"] = "Tautan audio tidak ditemukan dalam respons"
            return result
            
        if sound_url.startswith("/"):
            sound_url = "https://tikwm.com" + sound_url
            
        raw_title = music_info.get("title") or item.get("title") or f"sound_{video_id}"
        raw_author = music_info.get("author") or item.get("author", {}).get("unique_id", "unknown")
        
        sound_title = sanitize_filename(raw_title)
        author = sanitize_filename(raw_author)
        
        # Format penamaan file: [No]_[VideoID]_[Judul].mp3 (dipotong max 40 karakter agar path tidak terlalu panjang)
        short_title = sound_title[:40].strip()
        filename = f"{index:02d}_{video_id}_{short_title}.mp3"
        file_path = os.path.join(output_dir, filename)
        
        # Download data audio MP3
        audio_res = requests.get(sound_url, headers=headers, timeout=30)
        if audio_res.status_code == 200 and len(audio_res.content) > 1000:
            with open(file_path, "wb") as f:
                f.write(audio_res.content)
                
            result["status"] = "Berhasil"
            result["filename"] = filename
            result["file_path"] = file_path
            result["video_id"] = video_id
            result["sound_title"] = raw_title
            result["author"] = author
            result["duration_sec"] = music_info.get("duration") or item.get("duration")
            result["message"] = "OK"
        else:
            result["message"] = f"Data audio kosong atau korup (Ukuran: {len(audio_res.content)} bytes)"
            
    except requests.exceptions.RequestException as e:
        result["message"] = f"Gagal koneksi jaringan: {str(e)}"
    except Exception as e:
        result["message"] = f"Error tak terduga: {str(e)}"
        
    return result

---  
### 4. Eksekusi Crawling Semua Tautan ke Folder `hasil crawling sound`

In [9]:
crawling_results = []

print(f"[START] Memulai unduhan audio untuk {len(urls)} tautan TikTok...\n")

for i, url in enumerate(urls, 1):
    print(f"[{i:02d}/{len(urls):02d}] Memproses: {url} ...")
    res = download_tiktok_sound(url, output_dir=OUTPUT_DIR, index=i)
    crawling_results.append(res)
    
    if res['status'] == 'Berhasil':
        print(f"   ✓ Berhasil: {res['filename']} ({res['duration_sec']} detik)\n")
    else:
        print(f"   ✗ Gagal: {res['message']}\n")
        
    time.sleep(1)  # Jeda 1 detik antar request agar aman dan stabil

# Simpan rekapitulasi crawling ke CSV
df_log = pd.DataFrame(crawling_results)
df_log.to_csv(LOG_FILE, index=False, encoding='utf-8')

total_sukses = (df_log['status'] == 'Berhasil').sum()
total_gagal = len(urls) - total_sukses

print('='*50)
print('CRAWLING SELESAI!')
print(f'- Total Link      : {len(urls)}')
print(f'- Berhasil Diunduh: {total_sukses}')
print(f'- Gagal           : {total_gagal}')
print(f'- Log disimpan ke : {LOG_FILE}')
print(f'- File audio ada di folder: {OUTPUT_DIR}/')
print('='*50)

[START] Memulai unduhan audio untuk 32 tautan TikTok...

[01/32] Memproses: https://vt.tiktok.com/ZSgcVHh6q/ ...
   ✓ Berhasil: 01_7293914577102769414_original sound - endramansa.mp3 (20 detik)

[02/32] Memproses: https://vt.tiktok.com/ZSgcVPX2a/ ...
   ✓ Berhasil: 02_7650789279198612754_original sound - gilang.her.mp3 (29 detik)

[03/32] Memproses: https://vt.tiktok.com/ZSgcVMHwj/ ...
   ✓ Berhasil: 03_7595909134143278344_original sound - konco_boss.mp3 (112 detik)

[04/32] Memproses: https://vt.tiktok.com/ZSgcqXyV5/ ...
   ✓ Berhasil: 04_7546479203290500382_Stadium Rave.mp3 (53 detik)

[05/32] Memproses: https://vt.tiktok.com/ZSgcqcpej/ ...
   ✓ Berhasil: 05_7673002912519359752_original sound - kiorto2.mp3 (29 detik)

[06/32] Memproses: https://vt.tiktok.com/ZSgcq3h7h/ ...
   ✓ Berhasil: 06_7634181577748090134_original sound - clips06900.mp3 (185 detik)

[07/32] Memproses: https://vt.tiktok.com/ZSgcbJPKs/ ...
   ✓ Berhasil: 07_7527936063613013279_original sound - smitherenes.mp3 (12 